# Remote model checks
Hello World against (1) Muse Spark via Instructor (`lib.llm.extract_structured`) and (2) TypeSafe System One Jev (`TypeSafeClient.system_one`). Needs `LLM_API_KEY` + `TYPESAFE_API_KEY` in `etc/secrets/.env`.


In [3]:
%load_ext autoreload
%autoreload 2
import os
from lib.llm import get_llm_uri, get_default_model
print("LLM_URI:", get_llm_uri())
print("LLM_MODEL:", get_default_model())
print("LLM_API_KEY set:", bool(os.environ.get("LLM_API_KEY")))
print("TYPESAFE_API_KEY set:", bool(os.environ.get("TYPESAFE_API_KEY")))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
LLM_URI: https://api.meta.ai/v1
LLM_MODEL: muse-spark-1.3-contributor
LLM_API_KEY set: True
TYPESAFE_API_KEY set: True


In [4]:
from pydantic import BaseModel
from lib.llm import extract_structured

class HelloWorld(BaseModel):
    greeting: str

hello = extract_structured(HelloWorld, "Reply with greeting exactly 'Hello World'.")
print(hello)
print(hello.greeting)


2026-10-04 23:44:09,137 INFO lib.llm: Requesting structured LLM extraction model=muse-spark-1.3-contributor response_model=HelloWorld prompt_chars=42
2026-10-04 23:44:14,873 INFO lib.llm: Structured LLM extraction completed model=muse-spark-1.3-contributor response_model=HelloWorld elapsed_s=5.7


greeting='Hello World'
Hello World


In [5]:
from typesafe_sdk import Choice, TypeSafeClient

question = Choice(
    instructions="Does the state say hello?",
    criteria={"says_hello": "The state contains a greeting.", "other": "Anything else."},
)
with TypeSafeClient(model="jev-latest") as client:
    response = client.system_one("Hello World", {"hello": question})
print("model:", response.model)
answer = response.answers["hello"]
print("choice:", answer.choice)
print("probabilities:", answer.probabilities)


model: jev-1.13.0
choice: says_hello
probabilities: {'says_hello': 0.91, 'other': 0.09}
